<a href="https://colab.research.google.com/github/txellbalada/Reto_IA/blob/main/500Datos_Sinteticos_colomb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Reto IA: Telefonica challenge:
###Objetivo: entrenar un modelo capaz de distinguir entre un áudio real vs uno generado con IA

In [ ]:
import numpy as np
import pandas as pd
import librosa
import os
from scipy import stats
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
TIPO_DATASET = "sintetico"
ruta_carpeta = "/content/drive/MyDrive/Reto_Telefonica/Dataset_sint/Acento_Colombiano_F-M/"

In [ ]:
print("Carpeta configurada:")
print(ruta_carpeta)

print("\n¿Existe la carpeta?:", os.path.exists(ruta_carpeta))

if os.path.exists(ruta_carpeta):
    print("\nPrimeros archivos encontrados:")
    print(os.listdir(ruta_carpeta)[:10])

Carpeta configurada:
/content/drive/MyDrive/Reto_Telefonica/Dataset_sint/Acento_Colombiano_F-M/

¿Existe la carpeta?: True

Primeros archivos encontrados:
['CycleGAN-cof_02436_00052210902-cof_06136_015253.wav', 'CycleGAN-cof_02436_00301724065-cof_00610_006488.wav', 'CycleGAN-cof_02436_00733578342-cof_02484_012420.wav', 'CycleGAN-cof_02436_01680222370-cof_02484_015816.wav', 'CycleGAN-cof_02436_01718184551-cof_00610_016829.wav', 'CycleGAN-cof_02436_01680222370-cof_08784_007955.wav', 'CycleGAN-cof_02436_01718184551-com_00610_017757.wav', 'CycleGAN-cof_02436_00052210902-cof_00610_018463.wav', 'CycleGAN-cof_02436_00360598628-cof_08784_016788.wav', 'CycleGAN-cof_02436_01395350176-cof_06136_008334.wav']


In [ ]:
#Cargar audios
archivos = os.listdir(ruta_carpeta)
print(f"Se detectaron {len(archivos)} archivos en la carpeta.\n")

for archivo in archivos:
    if archivo.lower().endswith(".wav"):
        ruta_audio = os.path.join(ruta_carpeta, archivo)

        y, sr = librosa.load(ruta_audio, sr=16000)

        print(f"{archivo} cargado | duración: {len(y)/sr:.2f} segundos")

Se detectaron 500 archivos en la carpeta.

CycleGAN-cof_02436_00052210902-cof_06136_015253.wav cargado | duración: 6.07 segundos
CycleGAN-cof_02436_00301724065-cof_00610_006488.wav cargado | duración: 4.63 segundos
CycleGAN-cof_02436_00733578342-cof_02484_012420.wav cargado | duración: 6.25 segundos
CycleGAN-cof_02436_01680222370-cof_02484_015816.wav cargado | duración: 6.33 segundos
CycleGAN-cof_02436_01718184551-cof_00610_016829.wav cargado | duración: 5.23 segundos
CycleGAN-cof_02436_01680222370-cof_08784_007955.wav cargado | duración: 6.33 segundos
CycleGAN-cof_02436_01718184551-com_00610_017757.wav cargado | duración: 5.23 segundos
CycleGAN-cof_02436_00052210902-cof_00610_018463.wav cargado | duración: 6.07 segundos
CycleGAN-cof_02436_00360598628-cof_08784_016788.wav cargado | duración: 6.07 segundos
CycleGAN-cof_02436_01395350176-cof_06136_008334.wav cargado | duración: 5.23 segundos
CycleGAN-cof_02436_01622222208-cof_02484_017436.wav cargado | duración: 5.31 segundos
CycleGAN-co

In [ ]:
def extraer_etiquetas(nombre_audio, modo="auto"):
    """
    Extrae etiquetas binarias desde el nombre del audio.

    Devuelve un diccionario con:
    - label: 0 real / 1 sintético
    - genero_f: 1 mujer / 0 hombre
    - nacionalidad: colombiano / chileno / argentino
    - modelo generador: CycleGAN, Diff, StarGAN, TTS-Dif, TTS-StarGAN, TTS
    """

    nombre = nombre_audio.lower()

    etiquetas = {
        # Label principal
        "label": 0,

        # 🔥 Género (solo una columna)
        "genero_f": 0,

        # Nacionalidad
        "colombiano": 0,
        "chileno": 0,
        "argentino": 0,

        # Modelo generador
        "modelo_cyclegan": 0,
        "modelo_diff": 0,
        "modelo_stargan": 0,
        "modelo_tts_dif": 0,
        "modelo_tts_stargan": 0,
        "modelo_tts": 0
    }

    # -----------------------------
    # 1. Label principal
    # -----------------------------
    if modo == "real":
        etiquetas["label"] = 0
    elif modo == "sintetico":
        etiquetas["label"] = 1
    else:
        etiquetas["label"] = 1 if "-" in nombre_audio else 0

    # -----------------------------
    # 2. Nacionalidad + género
    # -----------------------------
    if "com" in nombre:
        etiquetas["colombiano"] = 1
        etiquetas["genero_f"] = 0

    elif "cof" in nombre:
        etiquetas["colombiano"] = 1
        etiquetas["genero_f"] = 1

    elif "clm" in nombre:
        etiquetas["chileno"] = 1
        etiquetas["genero_f"] = 0

    elif "clf" in nombre:
        etiquetas["chileno"] = 1
        etiquetas["genero_f"] = 1

    elif "arm" in nombre:
        etiquetas["argentino"] = 1
        etiquetas["genero_f"] = 0

    elif "arf" in nombre:
        etiquetas["argentino"] = 1
        etiquetas["genero_f"] = 1

    # -----------------------------
    # 3. Modelo generador
    # -----------------------------
    nombre_original = nombre_audio  # mantener mayúsculas

    if "TTS-StarGAN" in nombre_original:
        etiquetas["modelo_tts_stargan"] = 1
    elif "TTS-Dif" in nombre_original:
        etiquetas["modelo_tts_dif"] = 1
    elif "CycleGAN" in nombre_original:
        etiquetas["modelo_cyclegan"] = 1
    elif "StarGAN" in nombre_original:
        etiquetas["modelo_stargan"] = 1
    elif "Diff" in nombre_original:
        etiquetas["modelo_diff"] = 1
    elif "TTS" in nombre_original:
        etiquetas["modelo_tts"] = 1

    return etiquetas

In [ ]:
def generar_df_etiquetas(ruta_carpeta, tipo_dataset="auto"):
    """
    Genera un DataFrame con múltiples etiquetas:
    - id_audio: identificador secuencial (1, 2, 3, ...)
    - archivo: nombre original del archivo
    - label: real (0) vs sintético (1)
    - genero_f: mujer (1), hombre (0)
    - nacionalidad: colombiano / chileno / argentino
    - modelo generador
    """

    print(f"Explorando carpeta: {ruta_carpeta}")
    datos = []

    contador_id = 1

    for archivo in sorted(os.listdir(ruta_carpeta)):
        if archivo.lower().endswith(".wav"):
            nombre_audio = archivo.replace(".wav", "")

            etiquetas = extraer_etiquetas(nombre_audio, modo=tipo_dataset)

            fila = {
                "id_audio": contador_id,
                "archivo": archivo
            }

            fila.update(etiquetas)
            datos.append(fila)

            contador_id += 1

    df_etiquetas = pd.DataFrame(datos)

    print(f"DataFrame creado con {len(df_etiquetas)} registros.")
    print(f"Número de columnas: {df_etiquetas.shape[1]}")

    return df_etiquetas

In [ ]:
df_labels = generar_df_etiquetas(ruta_carpeta, tipo_dataset=TIPO_DATASET)
df_labels.head()

Explorando carpeta: /content/drive/MyDrive/Reto_Telefonica/Dataset_sint/Acento_Colombiano_F-M/
DataFrame creado con 500 registros.
Número de columnas: 13


,id_audio,archivo,label,genero_f,colombiano,chileno,argentino,modelo_cyclegan,modelo_diff,modelo_stargan,modelo_tts_dif,modelo_tts_stargan,modelo_tts
0,1,CycleGAN-cof_02436_00052210902-cof_00610_01846...,1,1,1,0,0,1,0,0,0,0,0
1,2,CycleGAN-cof_02436_00052210902-cof_02484_00619...,1,1,1,0,0,1,0,0,0,0,0
2,3,CycleGAN-cof_02436_00052210902-cof_06136_01525...,1,1,1,0,0,1,0,0,0,0,0
3,4,CycleGAN-cof_02436_00052210902-cof_08784_01299...,1,1,1,0,0,1,0,0,0,0,0
4,5,CycleGAN-cof_02436_00301724065-cof_00610_00648...,1,1,1,0,0,1,0,0,0,0,0


##  Mejor opción para TU problema (detección de voz fake)

Para detectar audio sintético, lo más importante es capturar:

- textura espectral (cómo suena la voz)  
- irregularidades (artefactos de IA)  
- dinámica (energía, variaciones)  

 Por eso, la mejor combinación es:


###  1. MFCC + Delta MFCC (OBLIGATORIO)

Son las más importantes.

Capturan:

- forma del espectro (timbre)  
- cambios en el tiempo  

✔ Detectan muy bien voces sintéticas  


###  2. Features de energía y estructura

- RMS → energía  
- ZCR → ruido/aspereza  

✔ Útiles para detectar artefactos de IA  



###  3. Features espectrales

- Spectral centroid → brillo  
- Bandwidth → dispersión  
- Rolloff → límite de energía  
- Flatness → tonal vs ruido  

✔ Clave para distinguir natural vs artificial  


###  4. Spectral contrast (MUY importante)

Esto muchas veces se subestima, pero:

 captura diferencias entre bandas de frecuencia  

✔ Muy útil para detectar:

- vocoders  
- modelos tipo GAN / TTS  


###  5. Estadísticas (CRÍTICO)

Esto es lo que mucha gente hace mal:

 NO usar los valores frame a frame  
 SÍ usar resumen:

- mean  
- std  
- min  
- max  
- median  
- q1  
- q3  

✔ Esto convierte el audio en una representación robusta  


## Entonces… ¿cuál es la mejor opción?

 EXACTAMENTE la que implementaste en el notebook nuevo:

✔ MFCC (13)  
✔ Delta MFCC  
✔ ZCR  
✔ RMS  
✔ Spectral features  
✔ Spectral contrast  
✔ Estadísticas completas  

In [ ]:
from scipy import stats

def resumir_feature(vector, prefijo, debug=False):
    """
    Resume un vector numérico con estadísticas descriptivas avanzadas.
    """

    vector = np.asarray(vector).astype(float)

    if len(vector) == 0:
        return {}

    resultado = {
        f"{prefijo}_mean": np.mean(vector),
        f"{prefijo}_std": np.std(vector),
        f"{prefijo}_min": np.min(vector),
        f"{prefijo}_max": np.max(vector),
        f"{prefijo}_median": np.median(vector),
        f"{prefijo}_q1": np.quantile(vector, 0.25),
        f"{prefijo}_q3": np.quantile(vector, 0.75),
        f"{prefijo}_skew": stats.skew(vector),
        f"{prefijo}_kurtosis": stats.kurtosis(vector),
        f"{prefijo}_mode": stats.mode(vector, keepdims=True)[0][0],
        f"{prefijo}_iqr": stats.iqr(vector)
    }

In [ ]:
#Función de extracción de features
def extraer_features(ruta_audio, sr_objetivo=16000, n_mfcc=13):
    """
    Extrae features de un audio y devuelve un diccionario.
    """

    y, sr = librosa.load(ruta_audio, sr=sr_objetivo)

    # Recorte de silencios
    y, _ = librosa.effects.trim(y)

    if len(y) < 512:
        return None

    features = {}

    # -------------------
    # META
    # -------------------
    features["duracion_seg"] = len(y) / sr

    # -------------------
    # ZCR
    # -------------------
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.update(resumir_feature(zcr, "zcr"))

    # -------------------
    # RMS (librosa)
    # -------------------
    rms = librosa.feature.rms(y=y)[0]
    features.update(resumir_feature(rms, "rms"))

    # -------------------
    # RMSE manual
    # -------------------
    rmse_manual = np.sqrt(np.mean(y**2))
    features["rmse_manual"] = rmse_manual

    # -------------------
    # TEMPO
    # -------------------
    tempo = librosa.beat.tempo(y=y, sr=sr)[0]
    features["tempo"] = tempo

    # -------------------
    # MFCC
    # -------------------
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    for i in range(n_mfcc):
        features.update(resumir_feature(mfcc[i], f"mfcc_{i+1}"))

    # -------------------
    # DELTA MFCC
    # -------------------
    delta_mfcc = librosa.feature.delta(mfcc)
    for i in range(n_mfcc):
        features.update(resumir_feature(delta_mfcc[i], f"delta_mfcc_{i+1}"))

    # -------------------
    # DELTA-DELTA MFCC
    # -------------------
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)
    for i in range(n_mfcc):
        features.update(resumir_feature(delta2_mfcc[i], f"delta2_mfcc_{i+1}"))

    # -------------------
    # SPECTRAL FEATURES
    # -------------------
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    features.update(resumir_feature(centroid, "centroid"))

    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    features.update(resumir_feature(bandwidth, "bandwidth"))

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        features.update(resumir_feature(contrast[i], f"contrast_{i+1}"))

    flatness = librosa.feature.spectral_flatness(y=y)[0]
    features.update(resumir_feature(flatness, "flatness"))

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    features.update(resumir_feature(rolloff, "rolloff"))

    # -------------------
    # PRINT FINAL
    # -------------------
    print(f"🎧 {os.path.basename(ruta_audio)}")
    print(f"   - Features extraídas: {len(features)}")
    print(f"   - Tempo: {tempo:.2f}")
    print(f"   - RMSE manual: {rmse_manual:.5f}")

    return features

In [ ]:
print("\n🔎 Probando extracción de features con un audio...\n")

for archivo in sorted(os.listdir(ruta_carpeta)):
    if archivo.lower().endswith(".wav"):
        ruta_audio_prueba = os.path.join(ruta_carpeta, archivo)

        try:
            ejemplo_features = extraer_features(ruta_audio_prueba)

            if ejemplo_features is None:
                print(f"⚠️ Audio omitido (muy corto): {archivo}")
                continue

            print("\n📊 Resultado:")
            print(f"   - Audio: {archivo}")
            print(f"   - Nº features: {len(ejemplo_features)}")

            # 🔥 Mostrar algunas features clave
            print("\n🔑 Ejemplo de features:")
            for k, v in list(ejemplo_features.items())[:5]:
                print(f"   {k}: {v}")

            # 🔥 Verificar si hay valores problemáticos
            valores = list(ejemplo_features.values())
            n_nan = sum(np.isnan(valores))
            n_inf = sum(np.isinf(valores))

            print("\n🧪 Control de calidad:")
            print(f"   - NaN: {n_nan}")
            print(f"   - Inf: {n_inf}")

            break

        except Exception as e:
            print(f"❌ Error con {archivo}: {e}")
            continue


🔎 Probando extracción de features con un audio...



/tmp/ipykernel_5188/2322890922.py:43: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = librosa.beat.tempo(y=y, sr=sr)[0]


🎧 CycleGAN-cof_02436_00052210902-cof_00610_018463.wav
   - Features extraídas: 575
   - Tempo: 156.25
   - RMSE manual: 0.09733

📊 Resultado:
   - Audio: CycleGAN-cof_02436_00052210902-cof_00610_018463.wav
   - Nº features: 575

🔑 Ejemplo de features:
   duracion_seg: 4.0
   zcr_mean: 0.1693173363095238
   zcr_std: 0.07817229830859039
   zcr_min: 0.05224609375
   zcr_max: 0.353515625

🧪 Control de calidad:
   - NaN: 0
   - Inf: 0


In [ ]:
registros = []

archivos_wav = sorted([a for a in os.listdir(ruta_carpeta) if a.lower().endswith(".wav")])

for idx, archivo in enumerate(tqdm(archivos_wav), start=1):
    ruta_audio = os.path.join(ruta_carpeta, archivo)
    nombre_audio = archivo.replace(".wav", "")

    try:
        etiquetas = extraer_etiquetas(nombre_audio, modo=TIPO_DATASET)
        feats = extraer_features(ruta_audio)

        if feats is None:
            print(f"Audio omitido por ser demasiado corto: {archivo}")
            continue

        fila = {
            "id_audio": idx,
            "archivo": archivo
        }

        fila.update(etiquetas)
        fila.update(feats)

        registros.append(fila)

    except Exception as e:
        print(f"Error procesando {archivo}: {e}")

  0%|          | 0/500 [00:00<?, ?it/s]/tmp/ipykernel_5188/2322890922.py:43: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = librosa.beat.tempo(y=y, sr=sr)[0]
  0%|          | 1/500 [00:00<01:48,  4.59it/s]

🎧 CycleGAN-cof_02436_00052210902-cof_00610_018463.wav
   - Features extraídas: 575
   - Tempo: 156.25
   - RMSE manual: 0.09733


  0%|          | 2/500 [00:00<01:49,  4.53it/s]

🎧 CycleGAN-cof_02436_00052210902-cof_02484_006194.wav
   - Features extraídas: 575
   - Tempo: 93.75
   - RMSE manual: 0.04725


  1%|          | 3/500 [00:00<01:49,  4.55it/s]

...

🎧 TTS-com_01523_00685957576.wav
   - Features extraídas: 575
   - Tempo: 170.45
   - RMSE manual: 0.09119


100%|█████████▉| 498/500 [02:06<00:00,  5.01it/s]

🎧 TTS-com_01523_00819269643.wav
   - Features extraídas: 575
   - Tempo: 125.00
   - RMSE manual: 0.07479
🎧 TTS-com_01523_00822655651.wav
   - Features extraídas: 575
   - Tempo: 93.75
   - RMSE manual: 0.08217


100%|██████████| 500/500 [02:06<00:00,  3.96it/s]

🎧 TTS-com_01523_00872110638.wav
   - Features extraídas: 575
   - Tempo: 104.17
   - RMSE manual: 0.09650
🎧 TTS-com_01523_00962087246.wav
   - Features extraídas: 575
   - Tempo: 125.00
   - RMSE manual: 0.08231


In [ ]:
print("\n Creando DataFrame...")

df = pd.DataFrame(registros)

print(" DataFrame creado")
print(f"   - Filas: {df.shape[0]}")
print(f"   - Columnas: {df.shape[1]}")

df.head()


 Creando DataFrame...
 DataFrame creado
   - Filas: 500
   - Columnas: 588


,id_audio,archivo,label,genero_f,colombiano,chileno,argentino,modelo_cyclegan,modelo_diff,modelo_stargan,...,rolloff_std,rolloff_min,rolloff_max,rolloff_median,rolloff_q1,rolloff_q3,rolloff_skew,rolloff_kurtosis,rolloff_mode,rolloff_iqr
0,1,CycleGAN-cof_02436_00052210902-cof_00610_01846...,1,1,1,0,0,1,0,0,...,1971.536037,789.0625,7562.5000,4183.59375,2037.109375,5710.937500,0.000705,-1.505250,1085.9375,3673.828125
1,2,CycleGAN-cof_02436_00052210902-cof_02484_00619...,1,1,1,0,0,1,0,0,...,2080.871242,554.6875,7539.0625,5492.18750,2152.343750,6203.125000,-0.451915,-1.419164,6304.6875,4050.781250
2,3,CycleGAN-cof_02436_00052210902-cof_06136_01525...,1,1,1,0,0,1,0,0,...,1965.394736,585.9375,7531.2500,4785.15625,1904.296875,5697.265625,-0.246978,-1.474301,1773.4375,3792.968750
3,4,CycleGAN-cof_02436_00052210902-cof_08784_01299...,1,1,1,0,0,1,0,0,...,1887.489328,601.5625,7468.7500,4789.06250,1978.515625,5585.937500,-0.261644,-1.407339,5585.9375,3607.421875
4,5,CycleGAN-cof_02436_00301724065-cof_00610_00648...,1,1,1,0,0,1,0,0,...,1878.109702,351.5625,7281.2500,4882.81250,2437.500000,5671.875000,-0.298326,-1.346327,1468.7500,3234.375000


In [ ]:
print("\n Distribución de labels:")
print(df["label"].value_counts())

print("\nProporciones:")
print(df["label"].value_counts(normalize=True))


 Distribución de labels:
label
1    500
Name: count, dtype: int64

Proporciones:
label
1    1.0
Name: proportion, dtype: float64


In [ ]:
#Revisar columnas
print("\n Información de columnas:")

print(f"Total columnas: {len(df.columns)}")

print("\nPrimeras 20 columnas:")
print(df.columns[:20])

print("\nÚltimas 20 columnas:")
print(df.columns[-20:])


 Información de columnas:
Total columnas: 588

Primeras 20 columnas:
Index(['id_audio', 'archivo', 'label', 'genero_f', 'colombiano', 'chileno',
       'argentino', 'modelo_cyclegan', 'modelo_diff', 'modelo_stargan',
       'modelo_tts_dif', 'modelo_tts_stargan', 'modelo_tts', 'duracion_seg',
       'zcr_mean', 'zcr_std', 'zcr_min', 'zcr_max', 'zcr_median', 'zcr_q1'],
      dtype='object')

Últimas 20 columnas:
Index(['flatness_min', 'flatness_max', 'flatness_median', 'flatness_q1',
       'flatness_q3', 'flatness_skew', 'flatness_kurtosis', 'flatness_mode',
       'flatness_iqr', 'rolloff_mean', 'rolloff_std', 'rolloff_min',
       'rolloff_max', 'rolloff_median', 'rolloff_q1', 'rolloff_q3',
       'rolloff_skew', 'rolloff_kurtosis', 'rolloff_mode', 'rolloff_iqr'],
      dtype='object')


In [ ]:
print("\n Información general del DataFrame:")
df.info()


 Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Columns: 588 entries, id_audio to rolloff_iqr
dtypes: float32(1), float64(574), int64(12), object(1)
memory usage: 2.2+ MB


In [ ]:
print("\n Estadísticas descriptivas:")
df.describe()


 Estadísticas descriptivas:


,id_audio,label,genero_f,colombiano,chileno,argentino,modelo_cyclegan,modelo_diff,modelo_stargan,modelo_tts_dif,...,rolloff_std,rolloff_min,rolloff_max,rolloff_median,rolloff_q1,rolloff_q3,rolloff_skew,rolloff_kurtosis,rolloff_mode,rolloff_iqr
count,500.000000,500.0,500.000000,500.0,500.0,500.0,500.000000,500.000000,500.000000,500.000000,...,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,250.500000,1.0,0.422000,1.0,0.0,0.0,0.180000,0.320000,0.180000,0.080000,...,1870.111740,511.921875,7239.218750,3589.734375,2118.132812,5232.843750,0.183033,-0.966562,2696.906250,3114.710938
std,144.481833,0.0,0.494373,0.0,0.0,0.0,0.384572,0.466943,0.384572,0.271565,...,213.235915,384.705693,327.829596,958.775987,510.565807,759.093836,0.506393,0.634523,1859.416679,814.784689
min,1.000000,1.0,0.000000,1.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,956.765792,85.937500,5164.062500,1675.781250,1015.625000,2498.046875,-1.143836,-1.700453,117.187500,488.281250
25%,125.750000,1.0,0.000000,1.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,1749.434610,171.875000,7083.984375,2849.609375,1778.808594,4849.609375,-0.188963,-1.403751,1328.125000,2720.703125
50%,250.500000,1.0,0.000000,1.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,1887.491265,437.500000,7312.500000,3353.515625,2062.500000,5416.015625,0.144847,-1.172475,2117.187500,3209.960938
75%,375.250000,1.0,1.000000,1.0,0.0,0.0,0.000000,1.000000,0.000000,0.000000,...,2007.260652,773.437500,7460.937500,4444.335938,2393.066406,5762.695312,0.578675,-0.752437,4289.062500,3679.199219
max,500.000000,1.0,1.000000,1.0,0.0,0.0,1.000000,1.000000,1.000000,1.000000,...,2418.111770,2101.562500,7843.750000,5875.000000,4468.750000,7140.625000,1.557499,2.207138,7375.000000,5021.484375


In [ ]:
print("\n Inspección de una fila completa:")

fila = df.iloc[0]

print(fila)
print("\nTotal de valores en esta fila:", len(fila))


 Inspección de una fila completa:
id_audio                                                            1
archivo             CycleGAN-cof_02436_00052210902-cof_00610_01846...
label                                                               1
genero_f                                                            1
colombiano                                                          1
                                          ...                        
rolloff_q3                                                  5710.9375
rolloff_skew                                                 0.000705
rolloff_kurtosis                                             -1.50525
rolloff_mode                                                1085.9375
rolloff_iqr                                               3673.828125
Name: 0, Length: 588, dtype: object

Total de valores en esta fila: 588


In [ ]:
print("\n Guardando dataset...")

nombre_csv = f"dataset_features_{TIPO_DATASET}.csv"
ruta_salida_csv = f"/content/drive/MyDrive/Reto_Telefonica/{nombre_csv}"

df.to_csv(ruta_salida_csv, index=False)

print(" Dataset guardado correctamente")
print(f" Ruta: {ruta_salida_csv}")
print(f" Shape final: {df.shape}")


 Guardando dataset...
 Dataset guardado correctamente
 Ruta: /content/drive/MyDrive/Reto_Telefonica/dataset_features_sintetico.csv
 Shape final: (500, 588)


In [ ]:
print("\n Preparando descarga...")

from google.colab import files
files.download(ruta_salida_csv)

print(" Descarga iniciada")


 Preparando descarga...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Descarga iniciada


In [ ]:
print("\n Guardando archivo auxiliar de etiquetas...")

df_labels = df[["id_audio", "archivo", "label"]]

ruta_labels = f"/content/drive/MyDrive/Reto_Telefonica/labels_{TIPO_DATASET}.csv"
df_labels.to_csv(ruta_labels, index=False)

print(" Archivo de labels guardado")
print(f" Ruta: {ruta_labels}")

df_labels.head()


 Guardando archivo auxiliar de etiquetas...
 Archivo de labels guardado
 Ruta: /content/drive/MyDrive/Reto_Telefonica/labels_sintetico.csv


,id_audio,archivo,label
0,1,CycleGAN-cof_02436_00052210902-cof_00610_01846...,1
1,2,CycleGAN-cof_02436_00052210902-cof_02484_00619...,1
2,3,CycleGAN-cof_02436_00052210902-cof_06136_01525...,1
3,4,CycleGAN-cof_02436_00052210902-cof_08784_01299...,1
4,5,CycleGAN-cof_02436_00301724065-cof_00610_00648...,1
